# Energy & Utilities Document Q&A (RAG)
Ask questions about energy/utilities PDF reports. Answers cite the source file and page.

**Run the cells top to bottom (press the play button on each).**

## Step 1: Install packages (takes ~2-3 min)

In [ ]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-chroma langchain-huggingface langchain-anthropic sentence-transformers chromadb pypdf

## Step 2: Add your API key
In Colab, click the **key icon** (Secrets) in the left sidebar → **Add new secret** → name: `ANTHROPIC_API_KEY`, value: your key → turn on **Notebook access**.

In [ ]:
import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
print("Key loaded")

## Step 3: Upload your PDF reports

In [ ]:
import os
from google.colab import files
os.makedirs("docs", exist_ok=True)
uploaded = files.upload()
for name, data in uploaded.items():
    with open(f"docs/{name}", "wb") as f:
        f.write(data)
print(os.listdir("docs"))

## Step 4: Ingest (read PDFs → split into chunks → embed → store in vector database)

In [ ]:
from pathlib import Path
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # free, runs locally

pages = []
for pdf in sorted(Path("docs").glob("*.pdf")):
    loaded = PyPDFLoader(str(pdf)).load()
    print(f"Loaded {pdf.name}: {len(loaded)} pages")
    pages.extend(loaded)

chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150).split_documents(pages)
print(f"Split into {len(chunks)} chunks")

embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
db = Chroma.from_documents(chunks, embeddings, persist_directory="chroma_db")
print("Vector database ready")

## Step 5: Set up question answering (retrieval + Claude)

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate

retriever = db.as_retriever(search_kwargs={"k": 4})
llm = ChatAnthropic(model="claude-sonnet-5-5")
prompt = ChatPromptTemplate.from_template(
    """You are an assistant answering questions about energy and utilities documents.
Use ONLY the context below. If the answer is not in the context, say you could not find it.

Context:
{context}

Question: {question}

Answer concisely."""
)
chain = prompt | llm

def _text(content):
    """Claude may return a list of blocks (reasoning + text); keep only the text."""
    if isinstance(content, str):
        return content
    return "".join(b.get("text", "") for b in content if isinstance(b, dict) and b.get("type") == "text")

def ask(question):
    docs = retriever.invoke(question)  # RETRIEVAL: find relevant chunks
    context = "\n\n".join(d.page_content for d in docs)
    reply = chain.invoke({"context": context, "question": question})  # GENERATION
    sources = sorted({f"{os.path.basename(d.metadata.get('source','?'))} (p. {d.metadata.get('page',0)+1})" for d in docs})
    print(_text(reply.content))
    print("\nSources:", ", ".join(sources))

## Step 6: Ask questions
Change the question and re-run. Try 10 different ones and save good examples for your README.

In [ ]:
ask("What are the main risks or challenges discussed in these documents?")

In [ ]:
ask("Your question here")